# 09 — Tier-1 SoTA: TTA + Multi-Scale + SWA Değerlendirmesi

**Amaç:** Teacher v4 checkpoint'ını *yeniden eğitmeden* Tier-1 tekniklerle iyileştirmek.

| Teknik | Beklenen kazanım | Referans |
|--------|------------------|----------|
| 8-transform geometrik TTA | +0.03–0.05 mF1 | TTA literatürü |
| Multi-scale (0.75/1.0/1.25) | +0.02–0.03 mF1 | çok ölçekli çıkarım |
| SWA (20 epoch fine-tune) | +0.01–0.03 mF1 | Izmailov et al. 2018 |

**Kural:** Bu notebook *incedir* — tüm kod `afetsonar` paketinden import edilir.
Değerlendirme `scripts/evaluate.py` ile, TTA `afetsonar.evaluation.tta_forward` ile,
SWA `afetsonar.training.AfetsonarTrainer` ile yapılır.

**Ön koşullar:**
1. Drive'da `afetsonar/` klasörü (checkpoint + xBD split CSV'leri)
2. Colab GPU runtime (H100/A100 önerilir)

Mevcut test sonuçları (baseline): Teacher mIoU_no_bg **0.424**, mF1 **0.640**.

In [ ]:
# GPU kontrolü
!nvidia-smi -L

In [ ]:
# Drive bağla
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Repoyu klonla (varsa güncelle) ve paketi kur
import os

if not os.path.exists("/content/calamitas-ai"):
    !git clone https://github.com/Runc-Dev/calamitas-ai.git /content/calamitas-ai
else:
    !cd /content/calamitas-ai && git pull

%cd /content/calamitas-ai
!pip install -q -e .

# Checkpoint'ların doğrulandığı transformers sürümü (708/708 anahtar eşleşir).
# Colab'ın önyüklü sürümü daha yeniyse SegFormer iç yapısı uyuşmaz.
!pip install -q "transformers==5.7.0"

In [ ]:
# ── Yol yapılandırması — kullanıcının gerçek Drive düzenine göre ──
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/AFETSONAR")  # klasör adı BÜYÜK harf
REPO_DIR   = Path("/content/calamitas-ai")

CKPT_DRIVE = DRIVE_ROOT / "checkpoints/teacher/teacher_v4_best_ema.pth"
CKPT_LOCAL = REPO_DIR / "checkpoints/teacher_v4_best_ema.pth"

# v3 split'leri = Tier 1+3 (9168 görüntü) — teacher_v4 bu split ile eğitildi.
# CSV'lerdeki görüntü yolları mutlak (/content/drive/MyDrive/AFETSONAR/...).
TEST_CSV   = DRIVE_ROOT / "data/splits/test_v3.csv"
TRAIN_CSV  = DRIVE_ROOT / "data/splits/train_v3.csv"  # (yalnızca SWA için)
VAL_CSV    = DRIVE_ROOT / "data/splits/val_v3.csv"    # (yalnızca SWA için)

# Checkpoint'ı yerel diske kopyala (Drive'dan okumak yavaştır)
CKPT_LOCAL.parent.mkdir(parents=True, exist_ok=True)
if not CKPT_LOCAL.exists():
    !cp "{CKPT_DRIVE}" "{CKPT_LOCAL}"

assert CKPT_LOCAL.exists(), "Checkpoint bulunamadı — CKPT_DRIVE yolunu kontrol edin"
assert TEST_CSV.exists(),   "test_v3.csv bulunamadı — TEST_CSV yolunu kontrol edin"
print("Hazır:", CKPT_LOCAL, f"({CKPT_LOCAL.stat().st_size/1e6:.0f} MB)")

## 1) Baseline — TTA'sız referans ölçüm

`scripts/evaluate.py` checkpoint mimarisini otomatik algılar (teacher/student).

In [ ]:
!python scripts/evaluate.py \
    --model "{CKPT_LOCAL}" \
    --test-csv "{TEST_CSV}" \
    --batch-size 8 \
    --output results/tier1_baseline.json
# Not: --image-size verilmedi — teacher otomatik 768px (natif çözünürlük) alır.
# 512'de değerlendirmek ~0.09 mF1 kaybettirir (doğrulandı: 0.551 vs 0.640).

## 2) 8-transform geometrik TTA

Görüntü başına 8 çıkarım (kimlik + 2 flip + 3 rotasyon + 2 flip·rotasyon),
olasılıklar ortalanır. Süre ≈ 8× baseline.

In [ ]:
!python scripts/evaluate.py \
    --model "{CKPT_LOCAL}" \
    --test-csv "{TEST_CSV}" \
    --batch-size 8 \
    --tta \
    --output results/tier1_tta.json

## 3) TTA + Multi-scale (0.75 / 1.0 / 1.25)

Görüntü başına 24 çıkarım (8 transform × 3 ölçek). En yüksek beklenen kazanım.

In [ ]:
!python scripts/evaluate.py \
    --model "{CKPT_LOCAL}" \
    --test-csv "{TEST_CSV}" \
    --batch-size 4 \
    --tta --tta-scales 0.75 1.0 1.25 \
    --output results/tier1_tta_ms.json
# 768 tabanlı ölçekler: 576 / 768 / 960 px

## 4) Karşılaştırma tablosu

In [ ]:
import json
import pandas as pd
from pathlib import Path
from afetsonar.config import CLASS_NAMES

runs = {
    "baseline":          "results/tier1_baseline.json",
    "tta_8x":            "results/tier1_tta.json",
    "tta_8x_multiscale": "results/tier1_tta_ms.json",
    "swa_tta":           "results/tier1_swa_tta.json",  # 5. adımdan sonra dolar
}

rows, missing = [], []
for name, path in runs.items():
    if not Path(path).exists():
        missing.append(f"{name}  ({path})")
        continue
    m = json.load(open(path))["metrics"]
    row = {"run": name,
           "mIoU_no_bg": round(m["miou_no_bg"], 4),
           "mF1": round(m["mf1"], 4),
           "accuracy": round(m["accuracy"], 4)}
    row.update({f"IoU_{CLASS_NAMES[i]}": round(v, 3)
                for i, v in enumerate(m["iou_per_class"])})
    rows.append(row)

if missing:
    print("Henüz üretilmemiş sonuçlar (ilgili hücreleri çalıştırın):")
    for m_ in missing:
        print("  -", m_)

if not rows:
    raise RuntimeError(
        "Hiç sonuç dosyası yok. Önce 1. bölümdeki (baseline) ve 2. bölümdeki "
        "(TTA) evaluate hücrelerini çalıştırıp bitmesini bekleyin — çıktının "
        "sonunda 'Results saved -> results/...' görünmeli."
    )

df = pd.DataFrame(rows).set_index("run")
if "baseline" in df.index:
    df["delta_mF1"] = (df["mF1"] - df.loc["baseline", "mF1"]).round(4)
df.to_csv("results/tier1_comparison.csv")
df

In [ ]:
# Sonuçları Drive'a yedekle
!mkdir -p "{DRIVE_ROOT}/outputs/tier1"
!cp results/tier1_* "{DRIVE_ROOT}/outputs/tier1/"
!ls -la "{DRIVE_ROOT}/outputs/tier1/"

## 5) (Opsiyonel) SWA fine-tune — Izmailov et al. 2018

Mevcut teacher v4'ten devam eder: düşük LR (1e-5), 20 epoch, son %20'de
ağırlık ortalaması. Experience replay catastrophic forgetting'i önler.

⚠️ H100'de ~2–3 saat sürer. Hazır olduğunuzda `RUN_SWA = True` yapın.

In [ ]:
RUN_SWA = False

if RUN_SWA:
    from afetsonar.training import AfetsonarTrainer

    trainer = AfetsonarTrainer(
        str(CKPT_LOCAL),
        mode="teacher",
        device="auto",
        checkpoints_dir="checkpoints",
    )
    result = trainer.resume_training(
        new_data_csv=str(TRAIN_CSV),
        val_csv=str(VAL_CSV),
        epochs=20,
        lr=1e-5,
        use_swa=True,
        batch_size=8,
        experiment_name="teacher_v5_swa",
    )
    SWA_CKPT = result["best_checkpoint"]
    print("En iyi checkpoint:", SWA_CKPT)
    print("En iyi val mIoU  :", result["best_val_miou"])

    # SWA modelini TTA ile değerlendir → tablo hücresini yeniden çalıştırın
    !python scripts/evaluate.py \
        --model "{SWA_CKPT}" \
        --test-csv "{TEST_CSV}" \
        --image-size 512 --batch-size 16 --tta \
        --output results/tier1_swa_tta.json

    # Yeni checkpoint'ı Drive'a yedekle
    !cp "{SWA_CKPT}" "{DRIVE_ROOT}/checkpoints/teacher/"

## Sonraki adımlar

1. `results/tier1_comparison.csv`'yi repoya commit'leyin
   (`git add results/tier1_comparison.csv && git commit && git push`).
2. En iyi konfigürasyonu `app.py`'de varsayılan yapın (TTA checkbox zaten var).
3. **Tier-2'ye geçiş:** Copy-Paste augmentation ile yeniden eğitim
   (`afetsonar.data.copy_paste.CopyPasteDataset` hazır) → beklenen +0.02–0.04 mF1.
4. Pseudo-labeling: teacher+TTA ile etiketsiz veriyi etiketle → student'ı yeniden damıt.